In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

import lightgbm as lgb
import xgboost as xgb
import shap

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, RandomizedSearchCV
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score

In [28]:
df = pd.read_csv("/Users/tam_pr/research/eda/data_processing/cleaned_df.csv")

In [29]:
X= df.copy()
X.drop(columns=["group_id","AA"], inplace=True)
y = df["AA"].copy()

In [30]:
splitter = GroupShuffleSplit(n_splits=1, test_size= 0.2, random_state= 1234)
train_indices, test_indices = next(splitter.split(X = X, y = y, groups = df["group_id"]))
X_train , X_test = X.iloc[train_indices], X.iloc[test_indices]
y_train , y_test = y.iloc[train_indices], y.iloc[test_indices]

In [31]:
group_id_train = df["group_id"].iloc[train_indices]

In [32]:
folds = list(GroupKFold(n_splits=5).split(X_train, y_train, groups= group_id_train))

In [33]:
etr_r2= []
etr_mae = []
all_mean_maes = []
all_mean_r2 = []

In [34]:
for fold_train_idx, fold_val_idx in folds: 
    X_fold_train = X_train.iloc[fold_train_idx]
    y_fold_train = y_train.iloc[fold_train_idx]

    X_fold_val = X_train.iloc[fold_val_idx]
    y_fold_val = y_train.iloc[fold_val_idx]

    model = ExtraTreesRegressor(random_state = 1234)

    model.fit(X_fold_train, y_fold_train)

    y_pred = model.predict(X_fold_val)

    etr_r2.append(r2_score(y_fold_val,y_pred))
    etr_mae.append(mean_absolute_error(y_fold_val,y_pred))


print("R^2 per fold: ", etr_r2)
print("Mean R^2", sum(etr_r2)/len(etr_r2))
all_mean_r2.append(sum(etr_r2)/len(etr_r2))

print("MAE per fold: ", etr_mae)
print("Mean MAE: ", sum(etr_mae)/ len(etr_mae))
all_mean_maes.append(sum(etr_mae)/ len(etr_mae))

etr_std_r2 = np.std(etr_r2, ddof=1)
etr_std_mae = np.std(etr_mae,ddof=1)

print("Standard deviation of the R^2 values: ", etr_std_r2)
print("Standard deviation of the MAE values: ", etr_std_mae)

R^2 per fold:  [0.875936526098078, 0.6687903028734876, 0.8424660735344391, 0.9332513166089005, 0.9919089284716671]
Mean R^2 0.8624706295173145
MAE per fold:  [3.2719945978134657, 6.387092372142682, 6.701328590720227, 2.7842412113214516, 1.0678295645904368]
Mean MAE:  4.0424972673176525
Standard deviation of the R^2 values:  0.12232922837675453
Standard deviation of the MAE values:  2.4285819329234486


In [35]:
cv = GroupKFold(n_splits=5)

In [36]:
r2_scores_2 = cross_val_score(
    model, X_train, y_train,
    groups = group_id_train, 
    cv = cv, 
    scoring ="r2"
)

mae_scores_2 = -cross_val_score(
    model, X_train, y_train, 
    groups = group_id_train, 
    cv = cv, 
    scoring="neg_mean_absolute_error"
)

print("R^2 per fold: ", r2_scores_2)
print("Mean R^2: ", r2_scores_2.mean())

print("MAE per fold: ", mae_scores_2)
print("Mean MAE: ", mae_scores_2.mean())

etr_std_r2 = np.std(r2_scores_2,ddof=1)
etr_std_mae = np.std(mae_scores_2,ddof=1)

print("Standard deviation of the R^2 values: ", etr_std_r2)
print("Standard deviation of the MAE values: ", etr_std_mae)

R^2 per fold:  [0.87593653 0.6687903  0.84246607 0.93325132 0.99190893]
Mean R^2:  0.8624706295173145
MAE per fold:  [3.2719946  6.38709237 6.70132859 2.78424121 1.06782956]
Mean MAE:  4.042497267317652
Standard deviation of the R^2 values:  0.12232922837675453
Standard deviation of the MAE values:  2.4285819329234486


In [37]:
results_etr = pd.DataFrame({
    "Model":["Extra Trees Regressor"]*5,
    "Fold": range(1, 6),
    "R^2": r2_scores_2,
    "MAE": mae_scores_2
})

print(results_etr.round(4).to_string(index=False))

                Model  Fold    R^2    MAE
Extra Trees Regressor     1 0.8759 3.2720
Extra Trees Regressor     2 0.6688 6.3871
Extra Trees Regressor     3 0.8425 6.7013
Extra Trees Regressor     4 0.9333 2.7842
Extra Trees Regressor     5 0.9919 1.0678


In [39]:
ExtraTreesRegressor().get_params().keys()

dict_keys(['bootstrap', 'ccp_alpha', 'criterion', 'max_depth', 'max_features', 'max_leaf_nodes', 'max_samples', 'min_impurity_decrease', 'min_samples_leaf', 'min_samples_split', 'min_weight_fraction_leaf', 'monotonic_cst', 'n_estimators', 'n_jobs', 'oob_score', 'random_state', 'verbose', 'warm_start'])

In [52]:
parameters = {
    "n_estimators" : [50, 100, 150, 200, 250],
    "max_depth" : [None, 2, 5, 10, 15, 20], 
    "min_samples_split" : [2, 4, 6, 8, 10] ,
    "min_samples_leaf" : [1, 3, 5, 7, 9, 11] , 
    "max_features" : ["sqrt", "log2", 1.0, 0.77], 
    "max_leaf_nodes" : [None, 10, 30, 50, 70, 90, 100], 
    "bootstrap" : [False, True]
}

In [53]:
random_search = RandomizedSearchCV(
    estimator=ExtraTreesRegressor(random_state=1234),
    param_distributions=parameters,
    n_iter=250,
    scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
    refit="mae",
    cv=folds,
    random_state=1234,
    n_jobs=-1,
    verbose = 1
)

In [54]:
random_search.fit(X_train, y_train)

print("Best parameters:")
print(random_search.best_params_)

print("\nBest CV MAE:", -random_search.best_score_)

cv_results = pd.DataFrame(random_search.cv_results_)

top_10 = cv_results.sort_values("rank_test_mae").head(10)[[
    "rank_test_mae",
    "mean_test_mae",
    "std_test_mae",
    "mean_test_r2",
    "std_test_r2",
    "params"
]].copy()

top_10["mean_test_mae"] = -top_10["mean_test_mae"]

top_10["std_test_mae_sample"] = (
    top_10["std_test_mae"] * (5 / 4) ** 0.5
)

top_10 = top_10.rename(columns={
    "rank_test_mae": "MAE Rank",
    "mean_test_mae": "Mean MAE",
    "std_test_mae": "Std MAE (population)",
    "std_test_mae_sample": "Std MAE (sample)",
    "mean_test_r2": "Mean R2",
    "std_test_r2": "Std R2",
    "params": "Parameters"
})

display(top_10.round(4))


Fitting 5 folds for each of 250 candidates, totalling 1250 fits


Best parameters:
{'n_estimators': 200, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_leaf_nodes': 70, 'max_features': 0.77, 'max_depth': 20, 'bootstrap': False}

Best CV MAE: 4.040951893261165


,MAE Rank,Mean MAE,Std MAE (population),Mean R2,Std R2,Parameters,Std MAE (sample)
150,1,4.0410,2.0769,0.8846,0.0848,"{'n_estimators': 200, 'min_samples_split': 6, ...",2.3221
155,2,4.0468,2.1503,0.8831,0.0876,"{'n_estimators': 250, 'min_samples_split': 6, ...",2.4041
91,3,4.0506,2.0159,0.8633,0.1208,"{'n_estimators': 50, 'min_samples_split': 4, '...",2.2538
36,4,4.0523,2.0875,0.8833,0.0865,"{'n_estimators': 150, 'min_samples_split': 6, ...",2.3339
146,5,4.0738,2.0925,0.8633,0.1201,"{'n_estimators': 50, 'min_samples_split': 4, '...",2.3395
238,6,4.0822,2.1736,0.8605,0.1142,"{'n_estimators': 100, 'min_samples_split': 4, ...",2.4302
7,6,4.0822,2.1736,0.8605,0.1142,"{'n_estimators': 100, 'min_samples_split': 4, ...",2.4302
57,8,4.0966,2.1171,0.8613,0.1239,"{'n_estimators': 250, 'min_samples_split': 6, ...",2.3669
21,9,4.1324,2.2198,0.8583,0.1168,"{'n_estimators': 200, 'min_samples_split': 4, ...",2.4819
120,10,4.1456,2.2162,0.8583,0.1169,"{'n_estimators': 200, 'min_samples_split': 2, ...",2.4777
